**Download** : 
1. [https://ollama.com/download](https://ollama.com/download)
2. [https://www.python.org/downloads/release/python-31210/](https://www.python.org/downloads/release/python-31210/)


| Model | | | 
| --- | --- |  --- | 
| qwen2:0.5b  | 6f48b936a09f    | 352 MB    |
| qwen:0.5b   | b5dc5e784f2a    | 394 MB    |
| qwen3:0.6b  | 7df6b6e09427    | 522 MB    |
| qwen2:1.5b  | f6daf2b25194    | 934 MB    |
| qwen:1.8b   | b6e8ec2e7126    | 1.1 GB    |
| qwen3:1.7b  | 8f68893c685c    | 1.4 GB    |
| qwen:4b     | d53d04290064    | 2.3 GB    |
| qwen3:4b    | 359d7dd4bcda    | 2.5 GB    |










In [ ]:
!pip install requests
#!ollama pull gemma3:270m-it-qat
#!ollama pull qwen2:0.5b   352 MB    
#!ollama pull qwen:0.5b    394 MB    
#!ollama pull qwen3:0.6b   522 MB    
#!ollama pull qwen2:1.5b   934 MB    
#!ollama pull qwen:1.8b    1.1 GB    
#!ollama pull qwen3:1.7b   1.4 GB    
#!ollama pull qwen:4b      2.3 GB    
#!ollama pull qwen3:4b     2.5 GB    

In [2]:
import requests
import json
import base64
import time
import os
from io import BytesIO

# ---------- Image helpers ----------
def image_to_base64_from_path(image_path):
    """Convert local image file to base64."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def image_to_base64_from_url(image_url):
    """Download image from URL and convert to base64."""
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode("utf-8")

def image_to_base64(image_input):
    """Return base64 string from path or URL."""
    if image_input.startswith(("http://", "https://")):
        return image_to_base64_from_url(image_input)
    else:
        return image_to_base64_from_path(image_input)

# ---------- Chat function ----------
def chat_with_thinking(prompt, model="qwen3:4b", image_input=None):
    """
    Streams a response from a thinking model.
    Optional image_input can be a local path or URL.
    Works for both text-only and multimodal models.
    """
    # Build message
    message = {"role": "user", "content": prompt}

    # If image provided, add it
    if image_input:
        try:
            image_b64 = image_to_base64(image_input)
            message["images"] = [image_b64]
        except Exception as e:
            print(f"Error loading image: {e}")
            return None, None

    payload = {
        "model": model,
        "messages": [message],
        "stream": True,
    }

    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        stream=True,
        timeout=300,
    )
    response.raise_for_status()

    start_time = time.time()
    first_thinking_time = None
    first_content_time = None
    thinking_text = ""
    content_text = ""

    print(f"\n=== Model: {model} ===")
    print(f"=== Prompt: {prompt} ===")
    if image_input:
        print(f"=== Image: {image_input} ===\n")
    else:
        print("=== Image: None ===\n")

    for line in response.iter_lines(decode_unicode=True):
        if not line:
            continue
        data = json.loads(line)
        msg = data.get("message", {})

        # Thinking
        if "thinking" in msg and msg["thinking"]:
            token = msg["thinking"]
            if first_thinking_time is None:
                first_thinking_time = time.time()
                print("--- THINKING START ---", end="\n", flush=True)
            print(token, end="", flush=True)
            thinking_text += token

        # Content
        if "content" in msg and msg["content"]:
            token = msg["content"]
            if first_content_time is None:
                first_content_time = time.time()
                if thinking_text:
                    print("\n--- THINKING END ---\n", end="", flush=True)
                else:
                    print("\n--- CONTENT START ---\n", end="", flush=True)
            print(token, end="", flush=True)
            content_text += token

        if data.get("done"):
            break

    total_time = time.time() - start_time

    print("\n\n=== TIMINGS ===")
    if first_thinking_time:
        print(f"TTFT (thinking): {first_thinking_time - start_time:.3f}s")
    if first_content_time:
        print(f"TTFT (content): {first_content_time - start_time:.3f}s")
    print(f"Total time: {total_time:.3f}s")
    print(f"Thinking chars: {len(thinking_text)}")
    print(f"Content chars: {len(content_text)}")

    return thinking_text, content_text


# ---------- Example usage ----------
if __name__ == "__main__":
    prompt = "Explain the difference between a research hypothesis, a mathematical model, and a simulation model."
    # 2.5 GB - qwen3:4b

    # --- Example 1: text-only thinking model ---
    chat_with_thinking(prompt, model="qwen3:4b")

    # --- Example 2: multimodal thinking model with image ---
    # image_path = r"C:\Users\Hulk\Pictures\test_image.jpg"
    # chat_with_thinking(prompt, model="qwen3-vl:8b-thinking", image_input=image_path)


=== Model: qwen3:4b ===
=== Prompt: Explain the difference between a research hypothesis, a mathematical model, and a simulation model. ===
=== Image: None ===

--- THINKING START ---
Okay, the user is asking about the difference between a research hypothesis, a mathematical model, and a simulation model. Let me start by recalling what each term means. 

First, a research hypothesis. That's something scientists propose to test. It's a statement about the relationship between variables. Like, "Increasing temperature speeds up chemical reactions." It's not proven yet; it's an educated guess. I should emphasize it's testable and part of the scientific method.

Next, mathematical models. These are equations that represent real-world systems. For example, Newton's laws for motion or the logistic growth equation for populations. They use math to describe how variables interact. Important to note they're abstract representations, not the real thing itself. They help predict outcomes but migh

In [8]:
import requests
import json
import base64
import time
import os
from IPython.display import display, Markdown, clear_output

# ---------- Image helpers (unchanged) ----------
def image_to_base64_from_path(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def image_to_base64_from_url(image_url):
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode("utf-8")

def image_to_base64(image_input):
    if image_input.startswith(("http://", "https://")):
        return image_to_base64_from_url(image_input)
    else:
        return image_to_base64_from_path(image_input)

# ---------- Notebook markdown streaming version ----------
def chat_with_thinking_markdown(prompt, model="qwen3:4b", image_input=None):
    """
    Streams response from a thinking model and updates a markdown cell.
    """
    message = {"role": "user", "content": prompt}
    if image_input:
        try:
            message["images"] = [image_to_base64(image_input)]
        except Exception as e:
            print(f"Error loading image: {e}")
            return

    payload = {
        "model": model,
        "messages": [message],
        "stream": True,
    }

    resp = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        stream=True,
        timeout=300,
    )
    resp.raise_for_status()

    # Timers
    start_time = time.time()
    first_thinking_time = None
    first_content_time = None
    thinking_text = ""
    content_text = ""

    # Create a display handle to update markdown
    handle = display(Markdown(""), display_id=True)

    for line in resp.iter_lines(decode_unicode=True):
        if not line:
            continue
        data = json.loads(line)
        msg = data.get("message", {})

        # Thinking
        if "thinking" in msg and msg["thinking"]:
            thinking_text += msg["thinking"]
            if first_thinking_time is None:
                first_thinking_time = time.time()

        # Content
        if "content" in msg and msg["content"]:
            content_text += msg["content"]
            if first_content_time is None:
                first_content_time = time.time()

        # Build markdown with current text
        md = f"## 🤔 Thinking\n\n{thinking_text}\n\n---\n\n## 💬 Answer\n\n{content_text}"
        handle.update(Markdown(md))

        if data.get("done"):
            break

    total_time = time.time() - start_time

    # Final update with timings
    timings = f"\n\n---\n**TTFT (thinking):** {first_thinking_time - start_time:.3f}s  \n" if first_thinking_time else ""
    timings += f"**TTFT (content):** {first_content_time - start_time:.3f}s  \n" if first_content_time else ""
    timings += f"**Total time:** {total_time:.3f}s"

    handle.update(Markdown(md + timings))

    return thinking_text, content_text

In [5]:

msg="""Explain the difference between a research hypothesis,
a mathematical model, and a simulation model.
Answer in 5 short bullet points and Step by Formula with citation"""

In [9]:
# 352 MB  - qwen2:0.5b    
a = chat_with_thinking_markdown(msg,model="qwen2:0.5b")

## 🤔 Thinking



---

## 💬 Answer

- A research hypothesis is a statement that proposes a specific set of facts or concepts that can be tested or falsified through experiments or observations.
- A mathematical model is a theoretical framework that describes how different variables are related and how the system behaves in a particular way.
- A simulation model is a computational model that simulates the behavior of a system over time or under conditions specified by the system's parameters.
- The difference between a research hypothesis and a mathematical model lies in their focus on testing or falsifying existing knowledge or hypotheses while a simulation model focuses on simulating the system's behavior or outcomes.
- The difference between a research hypothesis and a mathematical model is more about understanding the underlying principles and concepts that govern the system or process in question, while a simulation model is more about generating numerical data or data points that can be used to predict or confirm the system's behavior or outcomes.
- The difference between a research hypothesis and a mathematical model is less about the exact form or content of the concept being tested or falsified, but more about understanding the underlying principles and concepts that govern the system or process.**TTFT (content):** 0.000s  
**Total time:** 42.106s

In [ ]:
# 394 MB - qwen:0.5b
chat(msg,"qwen:0.5b")

In [ ]:
#  522 MB - qwen3:0.6b
chat(msg,"qwen3:0.6b")

In [ ]:
# 934 MB - qwen2:1.5b   
chat(msg,"qwen2:1.5b")

In [ ]:
# 1.1 GB - qwen:1.8b   
chat(msg,"qwen:1.8b")

In [ ]:
# 1.4 GB - qwen3:1.7b
chat(msg,"qwen3:1.7b")

In [ ]:
#  2.3 GB  - qwen:4b
chat(msg,"qwen:4b")

In [ ]:
# 2.5 GB - qwen3:4b
chat(msg,"qwen3:4b")